# ContextQA - Phase 2: Data Preprocessing, Tokenization & Span Alignment

In Extractive QA, the neural network does not generate words token by token. Instead, it solves a **span classification problem**:
- It receives an input sequence: `[CLS] Question [SEP] Context [SEP]`
- It predicts the probability distribution over all token indices for `start_position` and `end_position`.

In this notebook, we walk through the exact mechanics of Tokenization, `offset_mapping`, `sequence_ids`, and sliding windows.

In [ ]:
from transformers import AutoTokenizer
from src.preprocessing import prepare_train_features, prepare_validation_features

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Is Fast Tokenizer: {tokenizer.is_fast}")

## 1. Inspecting Subword Tokenization & Special Tokens

In [ ]:
question = "When was Stanford founded?"
context = "Stanford University was founded in 1885 by Leland Stanford."

inputs = tokenizer(question, context, return_offsets_mapping=True)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
sequence_ids = inputs.sequence_ids()

print(f"{'Idx':<4} {'Token':<15} {'SeqID':<6} {'Offset':<12}")
print("-" * 40)
for idx, (tok, seq, off) in enumerate(zip(tokens, sequence_ids, inputs["offset_mapping"])):
    print(f"{idx:<4} {tok:<15} {str(seq):<6} {str(off):<12}")

## 2. Character-to-Token Alignment with Sliding Windows

In [ ]:
examples = {
    "id": ["ex_1"],
    "title": ["Stanford"],
    "context": [context],
    "question": [question],
    "answers": [{"text": ["1885"], "answer_start": [context.index("1885")]}]
}

features = prepare_train_features(examples, tokenizer, max_seq_length=128, doc_stride=32)

start_idx = features["start_positions"][0]
end_idx = features["end_positions"][0]
input_ids = features["input_ids"][0]

print(f"Target Start Token Index: {start_idx}")
print(f"Target End Token Index:   {end_idx}")

# Reconstruct the answer from predicted tokens (end_idx is inclusive)
decoded = tokenizer.decode(input_ids[start_idx : end_idx + 1]).strip()
print(f"Decoded Token Span:       '{decoded}'")
assert decoded == "1885", "Error in token span reconstruction!"